# Week 02: Config-Driven Multi-Site — ScraperFlow V2

**Goal:** Scrape a second, different site without copy-pasting the first script. One `Scraper` class, two `SiteConfig` entries, zero site-specific branching.

**Concepts:** `@dataclass` (fields, defaults, frozen, `__post_init__`), composition over inheritance, configuration as data vs. code, essential vs. coincidental duplication, Rule of Three

**Disciplines this week:** Python, Architecture, Skills (DevTools inspection)

---
## Overview

### What are we building in ScraperFlow this week?

V2 transforms ScraperFlow from a single-site scraper into a **config-driven, multi-site** system. Instead of hardcoded CSS selectors in `parser.py`, a `SiteConfig` dataclass captures what varies per site (selectors, site name, base URL), while a generic `Scraper` class owns what stays fixed (the fetch → parse → store shape).

### Which version does this map to?

**Version 2 — Config-Driven Multi-Site Scraping** (Chapter 6, `v0.2 — Multi-Site`)

### What concepts do we need to learn before implementing?

1. **`@dataclass`** — Python's built-in mechanism for creating structured data containers with minimal boilerplate
2. **Composition over inheritance** — why passing data *into* a class is better than subclassing when the variation is data-shaped
3. **Configuration as data vs. code** — separating *what varies* from *what stays fixed*
4. **Essential vs. coincidental duplication** — knowing *when* to abstract

### Which disciplines does this week cover?

- **Python** — `@dataclass`, composition
- **Architecture** — configuration as data, duplication analysis, Rule of Three
- **Skills** — DevTools site inspection, designing `SiteConfig`

---
## Concepts by Discipline

### Python Concepts

**Core (full depth):** `@dataclass` — the load-bearing concept this week. You need to understand how dataclasses work under the hood to design `SiteConfig` correctly.

**Supporting (lighter):** composition over inheritance — you need to understand *why* this pattern is the right fit, but the Python mechanics are just "pass an object as a parameter."

---
## CORE: `@dataclass`

### 1. Terminology

| Term | Definition |
|---|---|
| **Dataclass** | A class decorated with `@dataclass` that auto-generates `__init__`, `__repr__`, `__eq__`, and optionally other dunder methods from annotated class variables |
| **Field** | An annotated class variable in a dataclass that becomes a parameter in the generated `__init__` |
| **`field()`** | A function from `dataclasses` module used to customize individual field behavior (defaults, repr inclusion, comparison) |
| **`default_factory`** | A callable that produces a fresh default value for mutable defaults (lists, dicts) — avoids the mutable default argument trap |
| **`frozen`** | A dataclass parameter that makes instances immutable — any attempt to set an attribute after creation raises `FrozenInstanceError` |
| **`__post_init__`** | A special method called automatically after the generated `__init__` completes — used for validation, computed fields, or type coercion |
| **Mutable default trap** | The bug where all instances share the same default mutable object (e.g., `list`) because Python evaluates default arguments once at class definition time |

### 2. What, Why & Consequences

**What is it?**

`@dataclass` (introduced in Python 3.7 via [PEP 557](https://peps.python.org/pep-0557/)) is a class decorator that automatically generates special methods based on annotated class variables. It's designed for classes that are primarily containers for data — where the interesting part is *what data they hold*, not *what methods they have*.

**Why does it exist?**

Before dataclasses, writing a simple data-holding class required repetitive boilerplate:

```python
class SiteConfig:
    def __init__(self, name, base_url, selectors):
        self.name = name
        self.base_url = base_url
        self.selectors = selectors
    
    def __repr__(self):
        return f"SiteConfig(name={self.name!r}, base_url={self.base_url!r}, ...)"
    
    def __eq__(self, other):
        return (self.name == other.name and 
                self.base_url == other.base_url and 
                self.selectors == other.selectors)
```

Every field appears four times (annotation, `__init__` parameter, `self.x = x`, and in `__repr__`/`__eq__`). Adding a field means touching all four places. Dataclasses eliminate this — you declare the fields once, and the methods are generated.

**What goes wrong without it?**

- Boilerplate errors: forget to update `__eq__` when adding a field → silent bugs in comparisons
- Missing `__repr__` → debugging becomes "what is this object?" instead of actually debugging
- Mutable defaults: if you write `def __init__(self, tags=[])`, all instances share the same list — a classic Python trap
- No enforcement of immutability — any code can mutate config objects accidentally

### 3. How It Works (Internals)

#### Execution Flow

When Python encounters `@dataclass`, here's what happens step by step:

```
@dataclass                          ← Step 1: decorator called at class definition time
class SiteConfig:                   ← Step 2: Python creates the class object first
    name: str                       ← Step 3: these become class-level annotations
    base_url: str                       (stored in __annotations__)
    selectors: dict[str, str]  
                                    ← Step 4: @dataclass inspects __annotations__
                                    ← Step 5: generates __init__, __repr__, __eq__
                                    ← Step 6: attaches generated methods to the class
                                    ← Step 7: returns the modified class
```

**Key detail:** `@dataclass` reads `__annotations__` (a dict of `{field_name: type}`) to know what fields exist. The type annotations are **metadata for the decorator** — they do not enforce types at runtime. Writing `name: str` doesn't prevent `name=42`.

#### What gets generated

```
@dataclass
class Foo:
    x: int
    y: str = "hello"

# Python generates (approximately):
#
# def __init__(self, x: int, y: str = "hello"):
#     self.x = x
#     self.y = y
#
# def __repr__(self):
#     return f"Foo(x={self.x!r}, y={self.y!r})"
#
# def __eq__(self, other):
#     if other.__class__ is self.__class__:
#         return (self.x, self.y) == (other.x, other.y)
#     return NotImplemented
```

#### The `@dataclass()` decorator parameters

| Parameter | Default | Effect |
|---|---|---|
| `init` | `True` | Generate `__init__` |
| `repr` | `True` | Generate `__repr__` |
| `eq` | `True` | Generate `__eq__` (and `__ne__`) |
| `order` | `False` | Generate `__lt__`, `__le__`, `__gt__`, `__ge__` |
| `frozen` | `False` | Make instances immutable (`__setattr__` and `__delattr__` raise `FrozenInstanceError`) |
| `slots` | `False` | Generate `__slots__` (Python 3.10+) — saves memory, prevents dynamic attributes |
| `kw_only` | `False` | All fields become keyword-only (Python 3.10+) |

#### Field ordering rule

Fields without defaults must come **before** fields with defaults — just like function parameters. This is enforced at class definition time:

```python
# This FAILS:
@dataclass
class Bad:
    x: int = 0
    y: str       # TypeError: non-default argument 'y' follows default argument
```

#### The mutable default trap and `field(default_factory=...)`

```
# WRONG — all instances share the same list object:
@dataclass
class Bad:
    tags: list = []   # ValueError: mutable default not allowed

# RIGHT — each instance gets a fresh list:
@dataclass
class Good:
    tags: list = field(default_factory=list)
```

Dataclasses explicitly prevent the mutable default trap — if you assign a mutable value (list, dict, set) as a default, Python raises `ValueError` at class definition time and tells you to use `field(default_factory=...)`.

#### `__post_init__` lifecycle

```
SiteConfig(name="books", base_url="https://...", selectors={...})
    │
    ├── Generated __init__ runs:
    │     self.name = "books"
    │     self.base_url = "https://..."
    │     self.selectors = {...}
    │
    └── __post_init__(self) runs:          ← you define this
          validate base_url starts with http
          check selectors is non-empty
          compute any derived fields
```

`__post_init__` is called automatically at the end of the generated `__init__`. It receives `self` with all fields already set. Use it for:
- **Validation** (raise `ValueError` if data is invalid)
- **Coercion** (convert string to enum, normalize URLs)
- **Computed fields** (derive a value from other fields)

**Gotcha with `frozen=True`:** If the dataclass is frozen, you can't do `self.computed = value` in `__post_init__` because assignment is blocked. Use `object.__setattr__(self, 'computed', value)` instead — this bypasses the frozen check.

### 4. When to Use / When NOT to Use

**Use `@dataclass` when:**
- The class is primarily a container for structured data
- You want auto-generated `__init__`, `__repr__`, `__eq__`
- The "interesting" part is what data the class holds, not what methods it has
- You need a typed, self-documenting alternative to plain dicts
- **ScraperFlow this week:** `SiteConfig` — it holds selectors and metadata, it doesn't *do* anything

**Don't use `@dataclass` when:**
- The class is mostly behavior with minimal data → regular class
- You need runtime type validation and coercion → use `pydantic.BaseModel` (Week 7)
- You need an immutable, hashable lightweight container → consider `NamedTuple`
- The data is truly unstructured or has an unknown shape at design time → use a plain dict

| Alternative | When it's better |
|---|---|
| Plain `dict` | Schema unknown or highly dynamic |
| `NamedTuple` | Immutable, hashable, needs tuple unpacking |
| `pydantic.BaseModel` | Need runtime type validation, serialization, coercion |
| Regular class | Mostly behavior, data is incidental |
| `TypedDict` | Type hints for dicts consumed by external APIs |

### 5. ScraperFlow Connection

`SiteConfig` in `scraperflow/config.py` — a `@dataclass` holding everything that varies per site:
- `name: str` — site identifier for logging and output
- `base_url: str` — the site's base URL
- `selectors: dict[str, str]` — maps field names to CSS selectors

The `Scraper` class (in `scraperflow/scraper.py`) receives a `SiteConfig` instance and uses it to drive the generic fetch → parse → store flow. The parser reads selectors from config instead of hardcoding them.

This is **composition**: `Scraper` *has a* `SiteConfig`, not `BooksScraper` *is a* `BaseScraper`. The variation (which selectors to use) is data — it belongs in a data container, not in a class hierarchy.

### 6. Worked Example — `@dataclass`

In [1]:
from dataclasses import dataclass, field


@dataclass(frozen=True)
class SiteConfig:
    """Configuration for a single scraping target."""
    name: str
    base_url: str
    selectors: dict[str, str]  # {field_name: css_selector}
    description: str = ""

    def __post_init__(self):
        if not self.base_url.startswith("http"):
            raise ValueError(f"base_url must start with http, got: {self.base_url!r}")
        if not self.selectors:
            raise ValueError("selectors cannot be empty")


# Two configs — zero code duplication, zero inheritance
books_config = SiteConfig(
    name="books_to_scrape",
    base_url="https://books.toscrape.com",
    selectors={
        "title": ".product_main h1",
        "price": ".price_color",
    },
    description="Book listings from Books to Scrape",
)

news_config = SiteConfig(
    name="example_news",
    base_url="https://news.example.com",
    selectors={
        "title": "h1.headline",
        "date": ".publish-date",
        "body": "article.content",
    },
)

print(books_config)
print(f"Books selectors: {books_config.selectors}")
print(f"News selectors:  {news_config.selectors}")
print(f"Equal? {books_config == news_config}")

# Frozen — this would raise FrozenInstanceError:
# books_config.name = "something_else"

SiteConfig(name='books_to_scrape', base_url='https://books.toscrape.com', selectors={'title': '.product_main h1', 'price': '.price_color'}, description='Book listings from Books to Scrape')
Books selectors: {'title': '.product_main h1', 'price': '.price_color'}
News selectors:  {'title': 'h1.headline', 'date': '.publish-date', 'body': 'article.content'}
Equal? False


### Worked Example — `field()` and `default_factory`

In [8]:
from dataclasses import dataclass, field


@dataclass
class ScrapeResult:
    url: str
    title: str
    # Mutable defaults MUST use field(default_factory=...)
    extra_fields: dict[str, str] = field(default_factory=dict)
    tags: list[str] = field(default_factory=list)
    success: bool = True


# Each instance gets its own fresh dict and list
r1 = ScrapeResult(url="https://a.com", title="Page A")
r2 = ScrapeResult(url="https://b.com", title="Page B")

r1.extra_fields["author"] = "Alice"
r1.tags.append("fiction")

print(f"r1 extras: {r1.extra_fields}")  # {'author': 'Alice'}
print(f"r2 extras: {r2.extra_fields}")  # {} — NOT affected by r1's mutation
print(f"r1 tags: {r1.tags}")            # ['fiction']
print(f"r2 tags: {r2.tags}")            # [] — independent

r1 extras: {'author': 'Alice'}
r2 extras: {}
r1 tags: ['fiction']
r2 tags: []


### Worked Example — `__post_init__` validation

In [3]:
from dataclasses import dataclass


@dataclass
class URLConfig:
    name: str
    base_url: str
    max_pages: int = 10

    def __post_init__(self):
        # Validation: catch bad data at construction time, not at use time
        if not self.base_url.startswith(("http://", "https://")):
            raise ValueError(
                f"base_url must start with http:// or https://, got: {self.base_url!r}"
            )
        if self.max_pages < 1:
            raise ValueError(f"max_pages must be >= 1, got: {self.max_pages}")


# Valid
config = URLConfig(name="test", base_url="https://example.com")
print(config)

# Invalid — caught immediately at construction
try:
    bad = URLConfig(name="bad", base_url="ftp://example.com")
except ValueError as e:
    print(f"Caught: {e}")

try:
    bad = URLConfig(name="bad", base_url="https://example.com", max_pages=0)
except ValueError as e:
    print(f"Caught: {e}")

URLConfig(name='test', base_url='https://example.com', max_pages=10)
Caught: base_url must start with http:// or https://, got: 'ftp://example.com'
Caught: max_pages must be >= 1, got: 0


---
## SUPPORTING: Composition Over Inheritance

### 1. What & Why

**Composition** means building behavior by combining objects — a class *receives* a collaborator as a parameter rather than *inheriting* from a parent. **Inheritance** means sharing behavior by extending a parent class — a child *is a* parent.

The key question: **is the variation between sites data-shaped or behavior-shaped?**

- If two sites differ only in *which CSS selectors to use* (data), a dataclass holding those selectors is the right tool — the behavior (fetch, select, extract text) is identical.
- If two sites differ in *how they fetch pages* (one uses HTTP, one uses a headless browser) — that's behavior-shaped variation, and a different pattern (Protocol + adapters, Week 5) is the right tool.

This week's variation is purely data-shaped. Inheritance would work, but it creates unnecessary coupling: every new site requires a new class, every class inherits the parent's full interface, and changing the parent risks breaking all children.

### 2. Key Patterns

```python
# COMPOSITION (right fit for V2):
class Scraper:
    def __init__(self, config: SiteConfig):  # receives data
        self.config = config
    
    def scrape(self, url: str) -> dict:
        html = fetch_page(url)
        return parse_generic(html, self.config.selectors)

# Adding a third site: just add a SiteConfig instance. Zero code changes.
```

```python
# INHERITANCE (wrong fit for V2):
class BaseScraper:
    def scrape(self, url: str) -> dict:
        html = fetch_page(url)
        return self.parse(html)  # subclass must override
    
    def parse(self, html: str) -> dict:
        raise NotImplementedError

class BooksScraper(BaseScraper):
    def parse(self, html: str) -> dict:
        # hardcoded selectors
        ...

# Adding a third site: write a whole new class. More code, tighter coupling.
```

| Factor | Composition | Inheritance |
|---|---|---|
| Adding a new site | Add a `SiteConfig` instance | Write a new class |
| Coupling | Loose — `Scraper` knows nothing about specific sites | Tight — each subclass is permanently bound to the parent |
| When behavior varies | Pass different objects (Strategy pattern) | Override methods |
| When data varies | Pass different data (this week) | Doesn't help — still need class per site |
| Testing | Pass a test config | Must instantiate a specific subclass |

### 3. ScraperFlow Connection

In `scraperflow/scraper.py`, the `Scraper` class receives a `SiteConfig` via its constructor. The parser function reads `config.selectors` to know which CSS selectors to apply. No `BooksScraper` or `NewsScraper` subclasses exist — adding a third site means only adding a third `SiteConfig` instance in `scraperflow/config.py`.

---
## Architecture Concepts

**Core (full depth):** Configuration as data vs. code, essential vs. coincidental duplication

**Supporting (lighter):** Rule of Three

---
## CORE: Configuration as Data vs. Code

### 1. Terminology

| Term | Definition |
|---|---|
| **Configuration as code** | Site-specific behavior embedded directly in source code — e.g., hardcoded selectors in `parse_article()` |
| **Configuration as data** | Site-specific details captured in a structured data container (dataclass, dict, JSON file) that is passed to generic code |
| **Parameterization** | Making a function/class generic by replacing hardcoded values with parameters |
| **Separation of concerns** | The principle that *what varies* (selectors) and *what stays fixed* (the scraping algorithm) should live in separate places |

### 2. What, Why & Consequences

**What is it?**

"Configuration as data" means extracting the parts that vary between contexts (sites, environments, runs) into a structured data object, leaving behind generic code that is parameterized by that data.

**V1's approach (configuration as code):**
```python
# parser.py — selectors are CODE, embedded in the function body
def parse_article(html, url):
    title_tag = soup.select_one(".product_main h1")  # hardcoded
    price_tag = soup.select_one(".price_color")       # hardcoded
```

**V2's approach (configuration as data):**
```python
# config.py — selectors are DATA, in a structured container
books = SiteConfig(name="books", selectors={"title": ".product_main h1", ...})

# parser.py — generic, parameterized by config
def parse_page(html, selectors):
    return {name: soup.select_one(sel).text for name, sel in selectors.items()}
```

**Why does it matter?**

- **Adding a site is additive, not invasive.** New `SiteConfig` instance, no code changes.
- **Testing is cleaner.** Pass a test config with known selectors and known HTML.
- **Debugging is easier.** Log the config object and you see exactly what selectors were used.
- **Future flexibility.** Config can eventually come from a file, a database, or an API — the code that consumes it doesn't change.

**What goes wrong without it?**

- Every new site means copying and modifying the parser function → N sites, N parsers
- Bug fixes must be applied to every copy independently
- Tests are tightly coupled to specific sites instead of testing the parsing *mechanism*

### 3. The Boundary in ScraperFlow

```
┌─────────────────────────────────┐     ┌─────────────────────────────────┐
│        WHAT VARIES              │     │        WHAT STAYS FIXED         │
│     (configuration as data)     │     │      (generic algorithm)        │
│                                 │     │                                 │
│  SiteConfig:                    │────▶│  Scraper:                       │
│    name = "books_to_scrape"     │     │    fetch_page(url)              │
│    base_url = "https://..."     │     │    parse(html, config.selectors)│
│    selectors = {                │     │    save(records, output_path)   │
│      "title": ".product_main h1"│     │                                 │
│      "price": ".price_color"    │     │  (zero site-specific strings)   │
│    }                            │     │                                 │
└─────────────────────────────────┘     └─────────────────────────────────┘
```

The arrow is a function parameter, not an inheritance relationship. `Scraper` receives a `SiteConfig` — it doesn't inherit from one or import one.

---
## CORE: Essential vs. Coincidental Duplication

### 1. Terminology

| Term | Definition |
|---|---|
| **Essential duplication** | Two pieces of code that represent the **same concept** — they look similar AND will always change together for the same reason |
| **Coincidental duplication** | Two pieces of code that **happen to look similar today** but represent different concepts — they will diverge as requirements change |
| **Rule of Three** | The heuristic that you should wait until you see the same pattern three times before abstracting — twice might be coincidence, three times is a pattern |

### 2. What, Why & Consequences

**What is it?**

When you see two pieces of code that look similar, the question isn't "should I DRY this up?" — it's "is this duplication essential or coincidental?"

**Essential duplication example (V2):**
```python
# These two parsers have the SAME shape: fetch → select → extract text
# The only difference is WHICH selectors they use — that's data, not logic.

def parse_books(html):    # same shape
    title = soup.select_one(".product_main h1").text
    price = soup.select_one(".price_color").text
    return {"title": title, "price": price}

def parse_news(html):     # same shape, different selectors
    title = soup.select_one("h1.headline").text
    date = soup.select_one(".publish-date").text
    return {"title": title, "date": date}
```
→ **Abstract it.** The shape is the same concept ("extract fields from HTML using CSS selectors"). The selectors are data that should be parameterized.

**Coincidental duplication example:**
```python
# These two functions look similar (both iterate and filter),
# but they represent DIFFERENT concerns that will diverge.

def filter_failed_urls(results):
    return [r for r in results if r["status"] != 200]

def filter_empty_records(records):
    return [r for r in records if r["title"]]
```
→ **Don't abstract it.** These will change for different reasons (network vs. data quality). Forcing them into a shared abstraction creates coupling where none should exist.

**The cost of getting it wrong:**
- Abstracting coincidental duplication → a shared function that needs `if` branches for each caller's special case → worse than the original duplication
- Not abstracting essential duplication → bug fixes applied to one copy but not the other → silent divergence

### 3. How to Decide

Ask three questions:

1. **Do these change for the same reason?** (If site A's selectors change, does site B's logic need to change too? No → coincidental.)
2. **Is the shape the same, with only data differing?** (Yes → essential. Extract the data into a parameter.)
3. **Would a shared abstraction need `if/elif` branches?** (Yes → you're forcing unrelated things together.)

### 4. ScraperFlow Connection

V1's `parse_article()` in `scraperflow/parser.py` has hardcoded selectors for Books to Scrape. When adding a second site, the shape of parsing (select elements, extract text, build a dict) is the same — but the selectors differ. This is essential duplication: the concept is "extract fields using CSS selectors," and the only thing that varies is *which* selectors. The fix is `SiteConfig` holding the selectors as data, not a second parser function with different hardcoded strings.

---
## SUPPORTING: Rule of Three

### 1. What & Why

The **Rule of Three** is a heuristic (attributed to Martin Fowler and others): don't abstract until you've seen the same pattern **three** times. Twice might be coincidence; three times is a signal.

### 2. Key Usage Pattern

- **First time:** just write it.
- **Second time:** notice the duplication but don't abstract yet — take a note.
- **Third time:** now abstract — you have enough evidence that the pattern is real.

**Why V2 acts at two, not three:**

The Rule of Three is a heuristic, not a law. V2 abstracts at the second site because:
- The variation is **purely data-shaped** — the extraction is trivially parameterizable
- The cost of abstraction is **very low** — a dataclass with a few fields
- The risk of being wrong is **very low** — if a third site needs behavior variation, `SiteConfig` can be extended or replaced without undoing the work

Compare this to Week 5 (Engine abstraction), where the Rule of Three is respected more strictly — because the cost and risk of abstracting an interface around fetch mechanisms is much higher.

### 3. ScraperFlow Connection

V2 uses two sites to validate that `SiteConfig`'s shape is correct. The handbook explicitly states: "abstracting from one example is guessing; a second real site makes the boundary honest." But the abstraction cost (one dataclass) is low enough that waiting for a third site would be over-cautious.

---
## Skills: DevTools Site Inspection

### Workflow for Designing a `SiteConfig`

Before writing any code for a new site, inspect it in browser DevTools:

1. **Open DevTools** → Elements tab (Cmd+Option+I on Mac)
2. **Use the element picker** (Cmd+Shift+C) to click on the data you want to extract
3. **Note the CSS selector path** — look for:
   - IDs (`#product-title`) — most specific, but may be dynamic
   - Classes (`.price-color`) — usually stable
   - Tag + class combinations (`h1.headline`) — good specificity
   - Structural selectors (`.product_main h1`) — stable if the page layout is stable
4. **Test your selector** in DevTools Console: `document.querySelector('.price-color')`
5. **Check multiple pages** — does the selector work on product page 1 AND product page 2?
6. **Record the selectors** as a dict for your `SiteConfig`

### Good vs. Bad Selectors

| Selector quality | Example | Why |
|---|---|---|
| Good | `.product_main h1` | Semantic class + tag, stable |
| Good | `#product-title` | ID, very specific |
| Fragile | `div > div > div > h1` | Depends on exact nesting depth |
| Fragile | `.css-1a2b3c` | Auto-generated class, changes on redeploy |
| Fragile | `body > main > section:nth-child(3) > div > h1` | Any layout change breaks it |

---
## Best Practices

### Python
- **Use `frozen=True`** for config objects that shouldn't change after creation — catches accidental mutation at the point it happens, not when a downstream function gets unexpected data
- **Use `field(default_factory=...)`** for mutable defaults — dataclasses enforce this, but understand *why* (the mutable default argument trap)
- **Validate in `__post_init__`** — catch invalid data at construction time, not at use time deep inside a scraping loop
- **Keep dataclasses focused on data** — methods are fine, but if the class has more methods than fields, it probably shouldn't be a dataclass

### Architecture
- **Ask "data or behavior?" before choosing composition or inheritance** — if the variation between cases is what data they hold, use composition (pass data in). If the variation is how they behave, consider inheritance or Protocol (Week 5).
- **Don't abstract coincidental duplication** — two things that look similar but change for different reasons should stay separate
- **Delete dead code** — when `SiteConfig` replaces hardcoded selectors, delete the old parser. Version control is your backup, not commented-out code.
- **Name your configs, don't number them** — `books_config` is debuggable, `config_1` is not

---
## Common Mistakes

### Python — `@dataclass` Mistakes

1. **Using a mutable default directly:** `tags: list = []` raises a `ValueError` in dataclasses (unlike regular classes, where it silently creates a shared-object bug). Fix: `tags: list = field(default_factory=list)`.

2. **Field ordering errors:** Putting a field with a default before one without. Dataclasses enforce the same rule as function parameters — required args first, defaults after.

3. **Trying to set attributes on a frozen dataclass in `__post_init__`:** `self.computed = value` raises `FrozenInstanceError`. Use `object.__setattr__(self, 'computed', value)` instead.

4. **Confusing `@dataclass` type annotations with runtime enforcement:** `name: str` doesn't prevent `SiteConfig(name=42)`. If you need runtime validation, use `pydantic` (Week 7) or explicit checks in `__post_init__`.

5. **Inheriting from a dataclass without understanding MRO:** Dataclass inheritance works but has subtle field-ordering gotchas — a parent with defaults followed by a child with required fields fails. Avoid unless you specifically need it.

### Architecture Mistakes

6. **Reaching for inheritance when the variation is data-shaped:** Writing `BooksScraper(BaseScraper)` when the only difference is which CSS selectors to use. This creates a class per site instead of a config per site.

7. **Abstracting coincidental duplication:** Forcing two similar-looking but conceptually unrelated pieces of code into a shared function, then needing `if/elif` branches — worse than the duplication.

8. **Building a config file loader before you need one:** Two hardcoded `SiteConfig` instances in Python are fine. Don't build YAML/JSON loading until a third or fourth site makes in-code configs unwieldy.

9. **Leaving V1's hardcoded parser as dead code:** "Just in case" code that's never called creates confusion about which path is actually used. Delete it — git has the history.

---
## Comparisons

### `@dataclass` vs. Alternatives

| Feature | `@dataclass` | `NamedTuple` | `pydantic.BaseModel` | Plain `dict` | Regular class |
|---|---|---|---|---|---|
| Mutable by default | Yes | No (immutable) | Yes | Yes | Yes |
| `frozen` option | Yes | Always frozen | `model_config = {'frozen': True}` | N/A | Manual |
| Runtime type validation | No | No | **Yes** | No | Manual |
| Auto `__init__` | Yes | Yes | Yes | N/A | No |
| Auto `__repr__` | Yes | Yes | Yes | Yes (sort of) | No |
| Auto `__eq__` | Yes | Yes | Yes | Yes | No |
| JSON serialization | Manual | Manual | **Built-in** | Built-in | Manual |
| Standard library | **Yes** | **Yes** | No (third-party) | **Yes** | **Yes** |
| Best for | Structured data containers | Immutable records, tuple replacement | Validated API data, external input | Unstructured / dynamic data | Behavior-heavy objects |

**Why `@dataclass` for `SiteConfig` this week:** Standard library (no dependency), provides `__init__`/`__repr__`/`__eq__` for free, supports `frozen` for immutability, and `__post_init__` for validation. We don't need `pydantic`'s runtime type coercion yet — that comes in Week 7 for `Record`.

### Composition vs. Inheritance

| Factor | Composition | Inheritance |
|---|---|---|
| Coupling strength | Low — connected by a parameter | High — permanently bound by class hierarchy |
| Adding a new variant | Add a data instance | Add a class |
| When variation is data | **Right fit** | Overkill |
| When variation is behavior | Possible (Strategy pattern) | Natural fit |
| Testing | Pass test data | Must subclass or mock |
| Reuse mechanism | "Has a" | "Is a" |
| Python convention | Preferred in most cases | Used for exceptions, GUI frameworks, ABCs |

---
## Real-World Use Cases

| Domain | How these concepts appear |
|---|---|
| **Web Scraping** | `SiteConfig` dataclasses drive generic scrapers across hundreds of sites — Scrapy's `Item` and spider settings are exactly this pattern |
| **Backend Development** | Django's `Settings` class, FastAPI's `BaseSettings` — configuration as data that parameterizes framework behavior |
| **Data Engineering** | Airflow DAG configs, dbt model configs — the transformation logic is generic, the config says *what* to transform |
| **AI/ML** | Model hyperparameters as dataclasses — same training loop, different configs per experiment |
| **APIs** | Request/response DTOs (Data Transfer Objects) — structured data containers for API payloads |
| **Testing** | Parameterized test fixtures — same test logic, different input data |
| **ScraperFlow V2** | `SiteConfig` dataclass in `scraperflow/config.py` — parameterizes `Scraper` so one class handles multiple sites with zero branching |

---
## Official References

### Python Documentation
- [`dataclasses` module](https://docs.python.org/3/library/dataclasses.html) — complete API reference
- [PEP 557 — Data Classes](https://peps.python.org/pep-0557/) — the original proposal, rationale, and design decisions
- [`dataclasses.field()`](https://docs.python.org/3/library/dataclasses.html#dataclasses.field) — field-level customization

### Architecture References
- [Martin Fowler — Rule of Three](https://martinfowler.com/bliki/RuleOfThree.html)
- [PyCon: "Stop Writing Classes" (Jack Diederich)](https://www.youtube.com/watch?v=o9pEzgHorH0) — composition over inheritance
- [*Architecture Patterns with Python*](https://www.cosmicpython.com/) — Percival & Gregory — composition and dependency injection

---
## Practice Exercises

### Easy
1. Create a `@dataclass` with three fields, one with a default. Instantiate it and print the auto-generated `__repr__`.
2. Create a frozen dataclass and verify that attempting to modify a field raises `FrozenInstanceError`.
3. Explain in one sentence: why does `@dataclass` forbid `tags: list = []` as a default?

### Medium
4. Write a `SiteConfig` dataclass with `__post_init__` validation that rejects invalid URLs. Test with both valid and invalid inputs.
5. Given two functions that parse different sites using hardcoded selectors (Books to Scrape and a news site), determine whether the duplication is essential or coincidental. Write a paragraph justifying your answer.
6. Refactor the two functions from exercise 5 into one generic function that takes a selector mapping as a parameter. What changed? What stayed the same?

### Hard
7. Design (don't implement) a `SiteConfig` for a real website you choose. Document: which fields vary per site? Which are truly optional? Would `frozen=True` be appropriate? Why or why not?
8. Write a short argument (3–5 sentences) for when inheritance *would* be the right choice over composition for a multi-site scraper. Name a specific scenario where behavior, not just data, varies between sites.
9. The Rule of Three says wait for three instances before abstracting. V2 abstracts at two. Write a paragraph defending this decision — what makes this case different from the general heuristic?

---
## Interview Questions

### Beginner
1. What is a Python `@dataclass`? What methods does it auto-generate?
2. What is the difference between `frozen=True` and a regular dataclass?
3. Why do you need `field(default_factory=list)` instead of `= []` in a dataclass?

### Intermediate
4. When would you choose a `@dataclass` over a `pydantic.BaseModel`? Give a specific scenario.
5. Explain composition over inheritance. Give an example of when each is the right choice.
6. What is the Rule of Three? When is it appropriate to abstract before seeing three instances?
7. You have two scrapers with nearly identical code but different CSS selectors. How would you eliminate the duplication? Why not use inheritance?

### Advanced
8. You're designing a config-driven scraper. Your teammate proposes a `BaseScraper` class hierarchy with a subclass per site. Argue for composition instead — what are the concrete trade-offs?
9. How do you distinguish essential duplication from coincidental duplication? Give examples of each and explain the cost of getting it wrong in both directions.
10. Your `SiteConfig` is currently a Python dataclass with hardcoded instances. At what point would you move to file-based config (JSON/YAML)? What signals would tell you it's time?

---
## Summary

### Key Takeaways

- **`@dataclass`** auto-generates `__init__`, `__repr__`, `__eq__` from annotated fields. Use `frozen=True` for immutability, `field(default_factory=...)` for mutable defaults, and `__post_init__` for validation.

- **Composition over inheritance** — when variation between cases is *data-shaped* (different selectors, not different behavior), pass a data object in rather than building a class hierarchy. Adding a new case = adding a data instance, not a new class.

- **Configuration as data vs. code** — extract what varies (selectors, URLs) into a structured container. Leave what stays fixed (fetch → parse → store) as generic, parameterized code.

- **Essential vs. coincidental duplication** — before DRYing code, ask: do these change for the same reason? Is the shape the same with only data differing? If yes → essential duplication, abstract it. If the similarity is superficial → leave it.

- **Rule of Three** — wait for three cases before abstracting, *unless* the cost of abstraction is very low and the variation is clearly data-shaped (as with `SiteConfig`).

- **This week's ScraperFlow deliverable:** One `Scraper` class + two `SiteConfig` instances + zero site-specific branching. The original hardcoded parser is deleted, not commented out.